In [ ]:
%load_ext autoreload
%autoreload 2

# Prepare Train&Valid Dataloader

In [ ]:
import yaml
from pprint import pprint

from torch.utils.data import DataLoader

from src.finebio import FineBioDataset
from src.train_utils import decode_clip, fix_random_seed

# load config
with open("configs/probing_atomic_operation.yaml", 'r') as f:
    cfg = yaml.safe_load(f)
dataset_cfg = cfg.get("dataset")
cls_cfg = cfg.get('classifier')
opt_cfg = cfg.get('opt')
assert opt_cfg is not None, 'Missing opt config.'
assert cls_cfg is not None, 'Missing classifier config.'
assert dataset_cfg is not None, "Unvalid config! Missing 'dataset' key."
pprint(cfg)
# build train dataset
train_dataset = FineBioDataset(split=["train"], **dataset_cfg)
rng_generator = fix_random_seed(cfg['init_rand_seed'], include_cuda=True)

# build train loader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=8,
    collate_fn=decode_clip,
    drop_last=True,
    generator=rng_generator
)

val_dataset = FineBioDataset(split=['valid'], **dataset_cfg)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    collate_fn=decode_clip,
    drop_last=False,
)

{'classifier': {'embed_dim': 1024, 'num_blocks': 4, 'num_heads': 16},
 'dataset': {'allow_clip_overlap': True,
             'frames_per_clip': 16,
             'json_file': 'data/annotations/annotation_all.json',
             'label2id_dir': 'data/annotations',
             'random_view': False,
             'sampling_rate': 1,
             'video_dir': ['data/FineBio/03 finebio_videos_fpv_all_w640',
                           'data/FineBio/10 '
                           'finebio_videos_tpv_all_w640/finebio_videos_w640']},
 'init_rand_seed': 2708,
 'opt': {'lr': 0.000425, 'warmup': 5, 'wd': 0.04}}


# Load V-JEPA 2

In [ ]:
from src.models import init_vjepa2

model, processor = init_vjepa2()

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

model total parameters: 325.97M


In [3]:
model.config.hidden_size

1024

# Build Attentive Classifier

In [ ]:
# load classifier
from src.classifiers import init_classifier

NUM_CLASSES = {k: v['num_classes'] for k,v in train_dataset.type_info.items()}
print(NUM_CLASSES)

classifier = init_classifier(
    num_verb_classes=NUM_CLASSES['verb'],
    num_mani_classes=NUM_CLASSES['manipulated'],
    num_affect_classes=NUM_CLASSES['affected'],
    **cls_cfg
)

{'verb': 10, 'manipulated': 35, 'affected': 36, 'atomic_operation': 244, 'hand': 2}


/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
classifier.embed_dim

1024

# Train probing layer

In [ ]:
import wandb
import time
import os
import torch
import torch.nn as nn

from src.losses import sigmoid_focal_loss
from src.optimizers import init_opt
from src.train_utils import train_one_epoch, valid_one_epoch

epochs = 40
lr = opt_cfg.get('lr')
wd = opt_cfg.get('wd')
warmup = opt_cfg.get('warmup', 5)
save_freq = 5
ckpt_dir = 'ckpt'
if os.path.exists(ckpt_dir):
    os.mkdir(ckpt_dir)
# criterion = nn.CrossEntropyLoss()
# criterion = sigmoid_focal_loss
optimizer, scheduler, wd_scheduler, _ = init_opt(
    params=classifier.parameters(), 
    iterations_per_epoch=len(train_dataset)//16,
    num_epochs=epochs,
    warmup=warmup,
    lr=lr,
    start_lr=lr,
    final_lr=lr,
    weight_decay=wd,
    final_weight_decay=wd,
)
logger = wandb.init(
    project='vjepa2-probing',
    name=f'debug-{int(time.time())}',
    mode='online',
    reinit='create_new'
)
for epoch in range(epochs):
    train_one_epoch(
        model=model,
        processor=processor,
        classifier=classifier,
        optimizer=optimizer,
        scheduler=scheduler,
        wd_scheduler=wd_scheduler,
        criterion=sigmoid_focal_loss,
        data_loader=train_loader,
        num_verb_classes=NUM_CLASSES['verb'],
        num_mani_classes=NUM_CLASSES['manipulated'],
        num_affect_classes=NUM_CLASSES['affected'],
        logger=logger,
    )
    # save model weights every save freq
    if (epoch+1)%save_freq == 0:
        save_path = os.path.join(ckpt_dir, f"{epoch+1}.pt")
        torch.save({
            "epoch": epoch,
            "classifier": classifier.state_dict(),
            # "optimizer": optimizer.state_dict(),
        }, save_path)
    
    valid_one_epoch(
        model=model,
        processor=processor,
        classifier=classifier,
        criterion=sigmoid_focal_loss,
        data_loader=val_loader,
        num_verb_classes=NUM_CLASSES['verb'],
        num_mani_classes=NUM_CLASSES['manipulated'],
        num_affect_classes=NUM_CLASSES['affected'],
        logger=logger,
    )
logger.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/tomtom/.netrc.
wandb: Currently logged in as: tangptnhan (tangptnhan-kyushu-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/4973 [00:00<?, ?it/s]/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
  8%|▊         | 418/4973 [14:03<2:33:30,  2.02s/it, loss=7.2659, verb_loss=3.0025, manip_loss=1.6132, aff_loss=2.6501]    